In [19]:
import os
from dotenv import load_dotenv
import warnings

warnings.filterwarnings("ignore")

# Load environment variables (e.g. DEEPSEEK_API_KEY)
load_dotenv()

from future_rag.indexing.embeddings import EmbeddingFactory
from future_rag.indexing.indexer import IndexBuilder
from future_rag.indexing.loader import IndexLoader
from future_rag.indexing.vectorstore import VectorStore
from future_rag.knowledge.chunker import DocumentChunker
from future_rag.knowledge.loaders import DocumentLoader
from future_rag.llms.factory import LLMFactory
from future_rag.pipelines.basic_rag import BasicRAG
from future_rag.retrieval.retriever import Retriever

print(" Setup and imports completed.")

 Setup and imports completed.


In [6]:
%%time
pdf_path = "../data/knowledge/pdf"
print(f"Loading documents from '{pdf_path}'...")

documents = DocumentLoader.load(pdf_path)
print(f"✅ Loaded {len(documents)} document(s).")

Loading documents from '../data/knowledge/pdf'...
✅ Loaded 22 document(s).
CPU times: total: 5.3 s
Wall time: 6.91 s


In [7]:
%%time
chunk_size = 1024
chunk_overlap = 256

print(f"Chunking documents with size={chunk_size}, overlap={chunk_overlap}...")

chunker = DocumentChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
nodes = chunker.split(documents)

print(f" Created {len(nodes)} chunks (nodes).")

Chunking documents with size=1024, overlap=256...
 Created 60 chunks (nodes).
CPU times: total: 1.31 s
Wall time: 1.91 s


In [8]:
%%time
print("Initializing embedding model...")

embedding_model = EmbeddingFactory.create()
print("✅ Embedding model initialized.")

Initializing embedding model...


2026-07-24 12:20:53,889 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-24 12:20:57,415 - INFO - Loaded 1 prompt with these keys: ['query']


✅ Embedding model initialized.
CPU times: total: 750 ms
Wall time: 4.04 s


In [9]:
%%time
from future_rag.indexing.vectorstore import VectorStore

db_path = "./data/vectorstores/chroma"
collection_name = "future_rag"

print(f"Connecting to ChromaDB at '{db_path}'...")

# 1. Instantiate your VectorStore manager
vector_store_manager = VectorStore(
    db_path=db_path,
    collection_name=collection_name,
)

# 2. Extract the LlamaIndex ChromaVectorStore instance
chroma_vector_store = vector_store_manager.connect()

print(" Connected to ChromaVectorStore successfully!")

Connecting to ChromaDB at './data/vectorstores/chroma'...
 Connected to ChromaVectorStore successfully!
CPU times: total: 359 ms
Wall time: 1.01 s


In [10]:
%%time
from future_rag.indexing.embeddings import EmbeddingFactory # adjust import path if needed
from future_rag.indexing.indexer import IndexBuilder
from future_rag.indexing.loader import IndexLoader

# 1. Initialize embedding_model using your factory
#embedding_model = EmbeddingFactory.create()

# 2. Check count directly via ChromaDB collection
existing_vector_count = chroma_vector_store._collection.count()

if existing_vector_count > 0:
    print(f"⚡ Found {existing_vector_count} existing vectors in ChromaDB.")
    print("Loading existing index using IndexLoader...")
    
    loader = IndexLoader(
        embedding_model=embedding_model,
        vector_store=chroma_vector_store,
    )
    index = loader.load()
    print(" Index successfully loaded from database!")

else:
    print(" Database is empty. Building new index using IndexBuilder...")
    
    builder = IndexBuilder(
        embedding_model=embedding_model,
        vector_store=chroma_vector_store,
    )
    index = builder.build(nodes=nodes)
    
    print(f" Index built and persisted! Total vectors: {chroma_vector_store._collection.count()}")

⚡ Found 60 existing vectors in ChromaDB.
Loading existing index using IndexLoader...
 Index successfully loaded from database!
CPU times: total: 93.8 ms
Wall time: 181 ms


In [20]:
%%time
print("Setting up Retriever and DeepSeek LLM...")

retriever = Retriever(index, similarity_top_k=3)
llm = LLMFactory.create(provider="deepseek")

print(" Retriever and LLM ready.")

Setting up Retriever and DeepSeek LLM...
 Retriever and LLM ready.
CPU times: total: 0 ns
Wall time: 5.52 ms


In [21]:
rag = BasicRAG(
    retriever=retriever,
    llm=llm,
)

print(" BasicRAG pipeline assembled.")

 BasicRAG pipeline assembled.


In [22]:

question = "What are the main principles of the FUTURE-AI guideline?"
print(f"Asking: '{question}'\n")

response = rag.query(question)

print("=== ANSWER ===")
print(response.response)




Asking: 'What are the main principles of the FUTURE-AI guideline?'



2026-07-24 12:31:36,682 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


=== ANSWER ===
Based solely on the provided context, the main principles of the FUTURE-AI guideline are fairness, universality, traceability, usability, robustness, and explainability.


In [14]:
question = "What is fairness in future AI guideline? how to ensure fairness in AI systems?"
print(f"Asking: '{question}'\n")

response = rag.query(question)

print("=== ANSWER ===")
print(response.response)

Asking: 'What is fairness in future AI guideline? how to ensure fairness in AI systems?'



2026-07-24 12:24:56,448 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


=== ANSWER ===
Based solely on the provided context, the FUTURE-AI guideline defines fairness as follows:

The fairness principle states that AI tools in healthcare should maintain the same performance across individuals and groups of individuals (including under-represented and disadvantaged groups). AI driven medical care should be provided equally for all citizens.

To ensure fairness in AI systems, the context provides three recommendations:

1.  **Fairness 1: Define sources of bias** - At the design phase, the interdisciplinary AI development team should identify possible types and sources of bias for their AI tool.
2.  **Fairness 2: Collect information on individual and data attributes** - To identify biases and apply measures for increased fairness, relevant attributes of the individuals should be collected, subject to informed consent and approval by ethics committees.
3.  **Fairness 3** - The context does not provide the text for a third recommendation, but it states that "thr

In [ ]:
question = "What are the main principles of the FUTURE-AI guideline?"

nodes = retriever.retrieve(question)

for i, node in enumerate(nodes, 1):
    print("=" * 80)
    print(f"Rank: {i}")
    print(f"Similarity Score: {node.score}")
    print(f"Metadata: {node.metadata}")
    print()
    print(node.text)
    print()

Rank: 1
Similarity Score: 0.6861970615752304
Metadata: {'page_label': '3', 'file_name': 'future-ai.pdf', 'file_path': 'd:\\FUTURE AI Workspace\\future-rag\\notebooks\\..\\data\\knowledge\\pdf\\future-ai.pdf', 'file_type': 'application/pdf', 'file_size': 1063171, 'creation_date': '2026-07-15', 'last_modified_date': '2026-02-04'}

RESEARCH METHODS AND REPORTINGRESEARCH METHODS AND REPORTING
mobile health, electronic health records, surgery, 
pathology). To develop the FUTURE-AI framework, 
we drew inspiration from the FAIR principles for data 
management, and defined concise recommendations 
organised according to six guiding principles—fairness, 
universality, traceability, usability, robustness, and 
explainability (fig 2).
Methods
FUTURE-AI is a structured framework that provides 
guiding principles and step-by-step recommendations 
for operationalising trustworthy and ethical AI in 
healthcare. This guideline was established through 
international consensus over a 24 month period usi